# 09 · Robots on a mesh

`Mesh("zenoh://", tenant=...)` makes every mounted robot a device other processes can discover, call, watch and
stop. Every write still lands in the robot's own safety chain. Here two meshes talk over loopback TCP, then a robot
in a second OS process joins.

![Two peers on one tenant: a bench mounts robots, a controller discovers, invokes and broadcasts estop](media/09/mesh_peers.svg)

In [1]:
import json
import os
import subprocess
import sys
import time
import uuid

import numpy as np
from PIL import Image, ImageDraw

from strands_robots import Robot, Simulation
from strands_robots.mesh import Mesh, RequestRefused, authority

import _nb

MEDIA = _nb.MEDIA / "09"
MEDIA.mkdir(parents=True, exist_ok=True)
PORT = 7450  # explicit endpoints switch multicast scouting off
TENANT = f"nb09-{uuid.uuid4().hex[:6]}"
ME, BENCH = f"zenoh://?listen=tcp/127.0.0.1:{PORT}", f"zenoh://?connect=tcp/127.0.0.1:{PORT}"

me = Mesh(ME, tenant=TENANT, operator="alice@laptop")
bench = Mesh(BENCH, tenant=TENANT, operator="bench@lab")
print(me)
print(bench)

Mesh('zenoh://?listen=tcp/127.0.0.1:7450', tenant='nb09-658988', mounted=[])
Mesh('zenoh://?connect=tcp/127.0.0.1:7450', tenant='nb09-658988', mounted=[])


## Mount

`bench.mount(robot, labels=...)` announces the robot with a presence token, answers its rpcs and publishes its state.
`Robot("so101", mesh=bench)` does the same in one line.

In [2]:
arm_world = Simulation("mujoco", render_size=(480, 360))
arm = Robot("so101", sim=arm_world)
arm_world.camera("cam", position=(0.55, -0.45, 0.4), target=(0.0, 0.0, 0.15), size=(480, 360))
g1 = Robot("g1", sim=Simulation())
print(bench.mount(arm, labels={"role": "arm", "bench": "left"}, state_hz=50))
print(bench.mount(g1, labels={"role": "humanoid", "bench": "left"}))

Device('so101', type='robot_arm', tenant='nb09-658988', labels={'role': 'arm', 'bench': 'left'})
Device('g1', type='humanoid', tenant='nb09-658988', labels={'role': 'humanoid', 'bench': 'left'})


## Discover

A selector is `device(*)` or `device(key:value,...)` over the labels plus `id`, `type` and `tenant`; a bare id means
`device(id:<id>)`.

In [3]:
t0 = time.perf_counter()
me.discover(min_count=2, timeout=5.0)
print(f"both devices visible to the controller after {(time.perf_counter() - t0) * 1000:.0f} ms")
for sel in ("device(*)", "device(type:robot_arm)", "device(role:humanoid)", "device(bench:left,type:humanoid)",
            "so101", "device(bench:right)"):
    print(f"{sel:36s} -> {[d.id for d in me.discover(sel)]}")
d = me.discover("so101")[0]
print("record:", d, "| controls:", [c["name"] for c in d.manifest["controls"]][:3], "...")

both devices visible to the controller after 1 ms
device(*)                            -> ['g1', 'so101']
device(type:robot_arm)               -> ['so101']
device(role:humanoid)                -> ['g1']
device(bench:left,type:humanoid)     -> ['g1']
so101                                -> ['so101']
device(bench:right)                  -> []
record: Device('so101', type='robot_arm', tenant='nb09-658988', labels={'role': 'arm', 'bench': 'left'}) | controls: ['shoulder_pan.pos', 'shoulder_lift.pos', 'elbow_flex.pos'] ...


## Invoke

`act` over the mesh runs `confirm_act` on the device; `seconds=` (up to 10) reaches the pose before it answers,
`steps=` advances that many control steps. A refusal from the device's chain comes back as the same
`RequestRefused`, message verbatim.

In [4]:
obs = me.invoke("so101", "act", action={"shoulder_pan": 0.3}, seconds=1.0)
print("act -> the device reads shoulder_pan =", round(arm.observe().joints["shoulder_pan"], 3))
try:
    me.invoke("so101", "act", action={"shoulder_pan": 99.0})
except RequestRefused as e:
    remote = str(e)
try:
    arm.confirm_act({"shoulder_pan": 99.0})
except RequestRefused as e:
    local = str(e)
print("over the mesh:", remote)
print("verbatim:", remote == local)
for rpc, kw in (("teleport", {"x": 1}), ("act", {"action": {"shoulder_pan": 0.1}, "seconds": 30.0})):
    try:
        me.invoke("so101", rpc, **kw)
    except (LookupError, TypeError, ValueError, RequestRefused) as e:
        print(f"{rpc}{sorted(kw)}: {type(e).__name__}: {e}")
print("rtt over loopback, 50 describe calls:")
ts = []
for _ in range(50):
    t0 = time.perf_counter()
    me.invoke("so101", "describe")
    ts.append((time.perf_counter() - t0) * 1000)
print(f"  p50 {np.percentile(ts, 50):.2f} ms, p90 {np.percentile(ts, 90):.2f} ms (this host, loaded)")

act -> the device reads shoulder_pan = 0.302
over the mesh: so101: shoulder_pan=99.0000 rad is outside its range [-1.9199, 1.9199]
verbatim: True
teleport['x']: LookupError: so101 has no rpc 'teleport' over the mesh; known: ['act', 'describe', 'estop', 'hold', 'lockout', 'observe', 'resume', 'stop']
act['action', 'seconds']: ValueError: act seconds= must be a number in (0, 10.0], got 30.0
rtt over loopback, 50 describe calls:
  p50 0.52 ms, p90 1.77 ms (this host, loaded)


## State over the mesh

The payload is the one `arm.subscribe("state", ...)` gives in-process; `seq` gaps are drops.

In [5]:
def wave(obs):
    t = obs.t
    return {"shoulder_pan": 0.8 * np.sin(1.5 * t), "shoulder_lift": -0.4 + 0.3 * np.sin(t),
            "elbow_flex": 0.6 + 0.4 * np.sin(1.2 * t), "wrist_flex": 0.5 * np.sin(2 * t), "gripper": 0.6}


got = []
unsubscribe = me.subscribe("so101", "state", got.append)
arm.run(wave, hz=30, seconds=1.0, realtime=True)
unsubscribe()
seqs = [g["seq"] for g in got]
print(f"{len(got)} samples; gaps in seq: {sum(b - a - 1 for a, b in zip(seqs, seqs[1:]))}")
print("last:", {k: round(v, 4) for k, v in got[-1].items() if k.startswith(("seq", "shoulder_pan", "elbow_flex.pos"))})

59 samples; gaps in seq: 0
last: {'seq': 62, 'shoulder_pan.pos': 0.1288, 'shoulder_pan.vel': -1.1188, 'shoulder_pan.effort': -0.4505, 'elbow_flex.pos': 0.896}


## `connect()`: a Robot whose body is on the other peer

Each tick of `run()` is one `observe` rpc and one `act` rpc; every write passes the device's chain.

In [6]:
arm.act({k: 0.0 for k in arm.action_keys}, seconds=0.5)
remote = me.connect("so101")
print(remote, type(remote.body).__name__)
frames = []


def caption(img, text, color=(0, 204, 96)):
    im = Image.fromarray(img)
    dr = ImageDraw.Draw(im)
    dr.rectangle([0, 0, im.width, 22], fill=(40, 41, 42))
    dr.text((8, 5), text, fill=color)
    return np.asarray(im)


def remote_wave(obs):
    if round(obs.t * 30) % 3 == 0:
        frames.append(caption(arm_world.render("cam"), f"me.connect('so101').run(wave)  t={obs.t:4.2f}s"))
    return wave(obs)


res = remote.run(remote_wave, hz=30, steps=120)
print(f"steps={res.steps} stopped_by={res.stopped_by!r}")
print("remote reads == device reads:", np.allclose(remote.observe().joints.positions, arm.observe().joints.positions))
Image.fromarray(frames[0]).save(MEDIA / "connect_run.gif", save_all=True, duration=100, loop=0, optimize=True,
                                append_images=[Image.fromarray(f) for f in frames[1:]])
print(f"media/09/connect_run.gif: {len(frames)} frames, {(MEDIA / 'connect_run.gif').stat().st_size / 1e6:.2f} MB")

Robot('so101', id='so101', remote on nb09-658988, dof=6) RemoteBody


steps=120 stopped_by='steps'
remote reads == device reads: True


media/09/connect_run.gif: 41 frames, 0.86 MB


![The SO-101 driven from the controller mesh through connect()](media/09/connect_run.gif)

## `broadcast("estop")`

It goes out on the tenant's `@estop` key (every mesh latches its matching devices on hearing it, no discovery
needed) and as an rpc to each discovered match, whose replies are the acks.

![broadcast estop: the tenant key latches every device, the per-device rpcs come back as acks](media/09/broadcast.svg)

In [7]:
task = arm.start(wave, hz=30, seconds=30, realtime=True)
time.sleep(0.5)
acked = me.broadcast("estop")
task.halt(2.0)
print("acked:", acked, "| arm run ended:", task.result.stopped_by)
for r in (arm, g1):
    print(r.id, "estopped:", r.estopped, "| lockout:", {k: v for k, v in me.invoke(r.id, "lockout").items() if k != "at"})

acked: ['g1', 'so101'] | arm run ended: estop
so101 estopped: True | lockout: {'engaged': True, 'epoch': 'b7db0a99f9c846618bc39a81dddd7d4b', 'by': 'alice@laptop'}
g1 estopped: True | lockout: {'engaged': True, 'epoch': 'c0f263a7bbe0422ea646facbd722d65b', 'by': 'alice@laptop'}


## Who may resume

Names are claims. Over zenoh, even on loopback, a device accepts a named resume only with `trust_names=True`; on
any network you do not control it wants an Ed25519 assertion signed for this stop and this device.

In [8]:
try:
    me.invoke("so101", "resume", approved_by=me.operator)
except RequestRefused as e:
    print("default authority:", e)
bench.resume_authority = authority.Authority(names=("safety-lead",), trust_names=True)
try:
    me.invoke("so101", "resume", approved_by="mallory")
except RequestRefused as e:
    print("mallory:", e)
me.invoke("g1", "resume", approved_by="safety-lead")
me.invoke("so101", "resume", approved_by=me.operator)  # the operator who stopped it
print("after safety-lead (g1) and alice (arm):", arm.estopped, g1.estopped)

default authority: so101: resume over this network needs operator keys (Mesh(resume_authority=Authority(public_keys=...)), or trust_names=True to accept claimed names)
mallory: so101: 'mallory' is not a resume authority for this stop (allowed: ['alice@laptop', 'safety-lead']). Resume as the operator who stopped it (broadcast/estop by=<operator>), name one in Mesh(resume_authority=Authority(names=[...])), or use operator keys (Authority(public_keys=...))
after safety-lead (g1) and alice (arm): False False


In [9]:
private, public = authority.keygen()
bench.resume_authority = authority.Authority(public_keys=(public,))
me.broadcast("estop", selector="so101")
try:
    me.invoke("so101", "resume", approved_by=me.operator)
except RequestRefused as e:
    print("unsigned:", e)
me.resume_key = private  # invoke() now fetches the lockout epoch and signs for it
me.invoke("so101", "resume", approved_by=me.operator)
print("signed resume -> latched:", arm.estopped, "| resumed_by:", arm.estop_latch.resumed_by)

unsigned: so101: resume needs an assertion signed by an operator key
signed resume -> latched: False | resumed_by: alice@laptop key=9539bdcb


## A robot in a second OS process

Same package, a new process with its own world, joining the controller's endpoint. It trusts the operator's public
key and nothing else.

In [10]:
peer_code = '''
import sys, time
from strands_robots import Robot
from strands_robots.mesh import Mesh, authority
url, tenant, key = sys.argv[1:4]
mesh = Mesh(url, tenant=tenant, operator="bench-b@lab", resume_authority=authority.Authority(public_keys=(key,)))
mesh.mount(Robot("so101", id="so101-b"), labels={"role": "arm", "bench": "right"})
print("ready", flush=True)
time.sleep(120)
'''
t0 = time.perf_counter()
peer = subprocess.Popen([sys.executable, "-c", peer_code, BENCH, TENANT, public], stdout=subprocess.PIPE, text=True)
assert peer.stdout.readline().strip() == "ready"
print(f"pid {peer.pid} ready after {time.perf_counter() - t0:.1f} s (imports + MuJoCo)")
t0 = time.perf_counter()
print(me.discover("device(bench:right)", min_count=1, timeout=5.0),
      f"found {(time.perf_counter() - t0) * 1000:.0f} ms after ready")
print("arms:", [d.id for d in me.discover("device(role:arm)")])

far = me.connect("so101-b")
res = far.run(wave, hz=30, steps=60)
print(f"connect('so101-b').run: steps={res.steps} stopped_by={res.stopped_by!r}, "
      f"shoulder_pan now {far.observe().joints['shoulder_pan']:+.3f}")
print("estop device(bench:right) acked by", me.broadcast("estop", selector="device(bench:right)"))
print("lockout in the other process:", me.invoke("so101-b", "lockout")["engaged"])
me.invoke("so101-b", "resume", approved_by=me.operator)  # signed with me.resume_key
print("after a signed resume:", me.invoke("so101-b", "lockout")["engaged"])

pid 60842 ready after 0.8 s (imports + MuJoCo)
[Device('so101-b', type='robot_arm', tenant='nb09-658988', labels={'role': 'arm', 'bench': 'right'})] found 1 ms after ready
arms: ['so101', 'so101-b']


connect('so101-b').run: steps=60 stopped_by='steps', shoulder_pan now -0.810
estop device(bench:right) acked by ['so101-b']
lockout in the other process: True
after a signed resume: False


In [11]:
peer.kill()
t0 = time.perf_counter()
while any(d.id == "so101-b" for d in me.discover()) and time.perf_counter() - t0 < 10:
    time.sleep(0.005)
print(f"SIGKILL -> gone from discover() after {(time.perf_counter() - t0) * 1000:.0f} ms (presence_ttl {me.presence_ttl:g} s)")
print("present:", [d.id for d in me.discover()], "| peer exit code:", peer.wait(5))

SIGKILL -> gone from discover() after 12 ms (presence_ttl 3 s)
present: ['g1', 'so101'] | peer exit code: -9


## `use_mesh`: the whole room as one agent tool

`me.tool` is generated from the manifests present. A refusal goes back to the model as text; the model may stop
but never resume.

In [12]:
tool = me.tool
schema = tool.tool_spec["inputSchema"]["json"]
print(tool.tool_spec["description"])
print("actions:", schema["properties"]["action"]["enum"])


def call(**inp):
    r = tool._tool_func({"toolUseId": "nb", "input": inp})
    return r["status"], r["content"][0]["text"][:160]


for inp in ({"action": "list", "selector": "device(role:arm)"},
            {"action": "act", "device_id": "so101", "act": {"shoulder_pan": 99.0}},
            {"action": "resume", "device_id": "so101", "approved_by": "alice@laptop"}):
    print(inp["action"], "->", *call(**inp))

Every device on the mesh (tenant nb09-658988). Selectors: device(*), device(role:arm,...).
- {"id": "g1", "type": "humanoid", "labels": {"bench": "left", "role": "humanoid"}, "rpcs": ["describe", "observe", "act", "stop", "hold", "estop", "resume"]}
- {"id": "so101", "type": "robot_arm", "labels": {"bench": "left", "role": "arm"}, "rpcs": ["describe", "observe", "act", "stop", "hold", "estop", "resume"]}
actions: ['list', 'describe', 'observe', 'act', 'run', 'stop', 'estop', 'resume']
list -> success [{"id": "so101", "device_type": "robot_arm", "labels": {"role": "arm", "bench": "left"}, "last_seen": 1793320318.9371169}]
act -> error RequestRefused: so101: shoulder_pan=99.0000 rad is outside its range [-1.9199, 1.9199]
resume -> error PermissionError: resume is the operator's, not an agent's: the operator clears so101 with mesh.invoke(id, 'resume', approved_by=...) or the dashboard


One real model turn (Amazon Bedrock, `STRANDS_MODEL_ID`).

In [13]:
from strands import Agent

agent = Agent(model=os.environ["STRANDS_MODEL_ID"], tools=[tool], callback_handler=None)
t0 = time.perf_counter()
reply = agent("List the arms on the mesh, move so101's shoulder_pan to 0.4 rad, then e-stop every device. "
              "Answer in two sentences.")
print(f"({time.perf_counter() - t0:.1f} s)", str(reply).strip())
for m in agent.messages:
    for b in m["content"]:
        if "toolUse" in b:
            print("  call:", json.dumps(b["toolUse"]["input"]))
print("arm estopped:", arm.estopped, "| shoulder_pan:", round(arm.observe().joints["shoulder_pan"], 3))
me.invoke("so101", "resume", approved_by=me.operator)  # the operator, in code
bench.resume_authority = authority.Authority(trust_names=True)
me.invoke("g1", "resume", approved_by=me.operator)
print("operator resumed:", not arm.estopped, not g1.estopped)

(7.2 s) The only arm on the mesh is so101, and its shoulder_pan moved to 0.4 rad (it read back as about 0.401 rad). Both devices, g1 and so101, are now e-stopped, and only an operator can clear that.
  call: {"action": "list", "selector": "device(role:arm)"}
  call: {"action": "act", "device_id": "so101", "act": {"shoulder_pan": 0.4}, "seconds": 2}
  call: {"action": "estop", "selector": "device(*)"}
arm estopped: True | shoulder_pan: 0.401
operator resumed: True True


## The same mesh over AWS IoT

`iot://fake?broker=<name>` is the in-process broker with IoT Core's Direct Messaging semantics; the calls are the
ones a real `iot://us-west-2` mesh takes.

In [14]:
fleet = Mesh("iot://fake?broker=nb09&client=bench", tenant=TENANT)
ops = Mesh("iot://fake?broker=nb09&client=ops", tenant=TENANT, operator="ops@cloud")
cloud_arm = Robot("so101", id="arm-1", sim=Simulation())
fleet.mount(cloud_arm, labels={"role": "arm"})
print(ops.discover(min_count=1, timeout=5.0))
ops.invoke("arm-1", "act", action={"shoulder_pan": 0.2}, steps=20)
print("estop acked by", ops.broadcast("estop"), "| latched:", cloud_arm.estopped)
ops.invoke("arm-1", "resume", approved_by="ops@cloud")  # in-process transport: names are trusted
print("resumed:", not cloud_arm.estopped)
fleet.close(); ops.close(); cloud_arm.sim.close()

[Device('arm-1', type='robot_arm', tenant='nb09-658988', labels={'role': 'arm'})]
estop acked by ['arm-1'] | latched: True
resumed: True


The deployment pieces are plain data. `greengrass.recipe()` and `kvs.pipeline()` need no AWS call:

In [15]:
from strands_robots.mesh.iot import greengrass, kvs

r = greengrass.recipe("1.0.0", "s3://bucket/strands_robots-1.0.0-py3-none-any.whl", tenant="lab",
                      robots=[{"name": "so101", "device_id": "arm-1"}])
print(r["ComponentName"], r["ComponentVersion"], "| run:", r["Manifests"][0]["Lifecycle"]["Run"]["Script"])
print("config:", r["ComponentConfiguration"]["DefaultConfiguration"])
print("kvs stream:", kvs.stream_name("lab", "arm-1", "wrist"))
print(kvs.pipeline(kvs.stream_name("lab", "arm-1", "wrist"), 640, 480, 15, region="us-west-2"))

com.strands.robots 1.0.0 | run: python3 -m strands_robots.mesh.iot.greengrass
config: {'tenant': 'lab', 'transports': ['zenoh://', 'iot://us-west-2'], 'robots': [{'name': 'so101', 'device_id': 'arm-1'}], 'state_hz': 10.0, 'shadow_hz': 0.5}
kvs stream: strands-lab-arm-1-wrist
appsrc name=src is-live=true format=time do-timestamp=true caps=video/x-raw,format=RGB,width=640,height=480,framerate=15/1 ! videoconvert ! x264enc tune=zerolatency speed-preset=ultrafast bitrate=1500 key-int-max=15 bframes=0 ! h264parse ! video/x-h264,stream-format=avc,alignment=au ! kvssink stream-name=strands-lab-arm-1-wrist aws-region=us-west-2 storage-size=128 fragment-duration=1000


```python
# needs AWS IoT credentials, device certificates and a Greengrass core
from strands_robots.mesh.iot import IotOptions, provision
cloud = Mesh("iot://us-west-2?client=arm-1&cert=~/.strands/iot/arm-1/cert.pem&key=~/.strands/iot/arm-1/key.pem",
             tenant="lab-a")
cloud.mount(Robot("so101"), device_id="arm-1", iot=IotOptions(shadow_hz=0.5))
provision.setup_tenant(iot, "lab-a", region="us-west-2", account="123456789012")
greengrass.deploy(gg, "arn:aws:iot:us-west-2:123456789012:thinggroup/strands-lab", "1.0.0")
kvs.publish_cameras(...)   # refuses by name without the GStreamer kvssink plugin
```

In [16]:
me.close()
bench.close()
arm_world.close()
g1.sim.close()
print("closed; peer gone:", peer.poll() is not None)

closed; peer gone: True
